# Epstein–Zin Lifecycle

A two-regime lifecycle savings model with health-dependent mortality and
Epstein–Zin preferences. The agent saves over annual periods starting at age 60,
faces a two-state health Markov chain, and transitions stochastically into a
terminal `dead` regime whose value is a bequest function. The model is an example
consumer block in the spirit of Atal, Fang, Karlsson & Ziebarth (2025, *JPE*
133(6), doi:10.1086/734781) — their richer model additionally includes insurance,
long-term care, and out-of-pocket medical spending — with the expected-utility
recursion replaced by Epstein–Zin.

[View source on GitHub](https://github.com/OpenSourceEconomics/pylcm/blob/main/src/lcm_examples/epstein_zin.py)

## The recursion and how it maps onto pylcm

The Epstein–Zin recursion separates the elasticity of intertemporal substitution
from risk aversion. With current consumption $c_t$ and continuation value
$V_{t+1}$:

```{math}
V_t = \Bigl[(1-\beta)\,c_t^{\rho} + \beta\,\mathrm{CE}_t^{\rho}\Bigr]^{1/\rho},
\qquad
\mathrm{CE}_t = \Bigl(\mathbb{E}_t\bigl[V_{t+1}^{\,1-\gamma}\bigr]\Bigr)^{1/(1-\gamma)}
```

where $\beta$ is the discount factor, $\rho = 1 - 1/\psi$ with $\psi$ the
elasticity of intertemporal substitution, and $\gamma$ is the coefficient of
relative risk aversion. When $\gamma = \rho$ the recursion collapses to expected
CRRA utility.

The mapping onto pylcm has four parts:

- **`utility`** returns $c_t$ directly — values stay in consumption units, keeping
  the power transform well-defined.
- **`H`** implements the outer Kreps–Porteus aggregator,
  `H = ((1 − β) · utility^ρ + β · E_next_V^ρ)^(1/ρ)`; its `E_next_V` argument
  receives the certainty equivalent already computed by the engine. It ships as
  `lcm.H_epstein_zin`, parametrized directly by the IES:
  `params["alive"]["H"]["intertemporal_elasticity_of_substitution"]` ($\psi$,
  with curvature $\rho = 1 - 1/\psi$ computed inside). `lcm.H_linear`
  (`U + β · E_next_V`, the default when no `H` is supplied) is the
  expected-utility counterpart.
- **`certainty_equivalent=PowerMean()`** implements $\mathrm{CE}_t$ with the power
  transform $g(v) = v^{1-\gamma}$. Its runtime parameter lives at
  `params["alive"]["certainty_equivalent"]["risk_aversion"]`.
- The **expectation** runs jointly over the health Markov chain and the alive/dead
  regime transition:
  $\mathrm{CE} = g^{-1}\!\bigl(\sum_r p_r\,\mathbb{E}_w[g(V'_r)]\bigr)$ with
  $g(v) = v^{1-\gamma}$. The engine handles the two-level expectation
  automatically given the stochastic regime transition and the stochastic health
  transition in `state_transitions`.

## Pitfalls

- **Positivity.** Power transforms require $V' > 0$ everywhere: with $\gamma > 1$,
  $g(0) = \infty$ propagates backward and corrupts the entire solution. Implicit
  zeros count too — a reachable target regime with no states contributes $0$ to
  the transformed sum, which is coherent only when $g(0) = 0$. Keep values in
  consumption units (`utility = c`, not `utility = log(c)`) and model death with
  an explicit wealth state and a strictly positive bequest — here `sqrt(wealth)`,
  with a consumption/wealth lower bound of 0.5 in the alive regime, so every
  reachable wealth is positive.
- **`risk_aversion = 1`** is the geometric-mean (log) limit of the power mean,
  $\exp(\mathbb{E}[\log V'])$; `PowerMean` handles it directly.
- **Solver restriction.** `certainty_equivalent` is supported only with the
  `GridSearch` solver. Passing `certainty_equivalent` together with a DC-EGM solver
  is rejected at model build.

## Run it

Build a 20-period model on fine wealth and consumption grids, then solve and
simulate 100 subjects under three preference configurations: expected utility
(the linear certainty equivalent, `certainty_equivalent=None`) and Epstein–Zin
with low and high risk aversion. The more risk-averse agent holds more wealth
against health and mortality risk.

In [ ]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

from lcm import PowerMean
from lcm_examples.epstein_zin import EZRegimeId, HealthStatus, get_model, get_params

n_periods = 20
survival_probs = tuple(np.linspace(0.97, 0.0, n_periods - 1))
grid_sizes = {"n_wealth_points": 60, "n_consumption_points": 60}
model_ez = get_model(
    certainty_equivalent=PowerMean(), n_periods=n_periods, **grid_sizes
)
model_eu = get_model(certainty_equivalent=None, n_periods=n_periods, **grid_sizes)

In [ ]:
n_subjects = 100
configs = {
    "expected utility": (model_eu, None),
    "Epstein–Zin, γ = 0.5": (model_ez, 0.5),
    "Epstein–Zin, γ = 5.0": (model_ez, 5.0),
}
mean_wealth_by_age = {}
for label, (m, risk_aversion) in configs.items():
    params = get_params(risk_aversion=risk_aversion, survival_probs=survival_probs)
    result = m.simulate(
        params=params,
        initial_conditions={
            "age": jnp.full(n_subjects, 60.0),
            "wealth": jnp.linspace(1.0, 10.0, n_subjects),
            "health": jnp.full(n_subjects, HealthStatus.good, dtype=jnp.int32),
            "regime_id": jnp.full(n_subjects, EZRegimeId.alive, dtype=jnp.int32),
        },
        period_to_regime_to_V_arr=None,
        log_level="off",
        seed=42,
    )
    df = result.to_dataframe()
    alive = df[df["regime_name"] == "alive"]
    mean_wealth_by_age[label] = alive.groupby("age")["wealth"].mean()

In [ ]:
styles = {
    "expected utility": {"color": "#999999", "width": 2.5, "dash": "dash"},
    "Epstein–Zin, γ = 0.5": {"color": "#457b9d", "width": 2.5},
    "Epstein–Zin, γ = 5.0": {"color": "#e63946", "width": 2.5},
}

fig = go.Figure()
for label, series in mean_wealth_by_age.items():
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series.to_numpy(),
            mode="lines",
            name=label,
            line=styles[label],
            showlegend=False,
        )
    )
    fig.add_annotation(
        x=series.index[-1],
        y=series.to_numpy()[-1],
        text=label,
        font={"color": styles[label]["color"]},
        xanchor="left",
        xshift=6,
        showarrow=False,
    )

fig.update_layout(
    title="Mean wealth of alive subjects over the lifecycle",
    xaxis_title="Age",
    yaxis_title="Mean wealth (alive subjects)",
    template="plotly_white",
    margin={"r": 190},
)
fig